# Monte Carlo Simulation: OLS vs Newey–West HAC under AR(4) Autocorrelation

Goal: Evaluate the performance of ordinary least squares (OLS) standard errors
compared to heteroskedasticity-and-autocorrelation consistent (HAC)
Newey–West estimators when residuals follow an AR(4) process.


In [58]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from scipy.stats import norm, chi2, t
from tqdm.notebook import trange, tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [59]:
# Simulation parameters
np.random.seed(2025)
R = 1000                # number of replications
T = 100                 # sample size
betas_true = np.array([0.0, 1.0, 0.5, -0.5])
phi = np.array([0.4, -0.2, 0.15, -0.05])  # AR(4) coefficients
bandwidths = [0, 1, 4, int(4 * (T/100)**(2/9))]  # HAC lag lengths


In [60]:
def simulate_ar4(T, phi, sigma=1.0):
    epsilon = np.random.normal(0, sigma, T)
    u = np.zeros(T)
    for t in range(4, T):
        u[t] = phi[0]*u[t-1] + phi[1]*u[t-2] + phi[2]*u[t-3] + phi[3]*u[t-4] + epsilon[t]
    return u

def one_replication(T, phi, betas, bandwidths):
    """
    Run one Monte Carlo replication comparing OLS and Newey-West estimators.

    Returns a dictionary with:
    betahat, sandwich variance, beta variance, confidence intervals, and p-values.
    """

    # --- 1️⃣ Simulate regressors ---
    X = np.column_stack([
        np.ones(T),
        np.random.normal(size=T),
        np.random.normal(size=T),
        np.random.normal(size=T)
    ])

    # --- 2️⃣ Simulate AR(4) errors ---
    u = simulate_ar4(T,phi)

    # --- 3️⃣ Generate dependent variable ---
    y = np.matmul(X,betas) + u

    # --- 4️⃣ Fit OLS model ---
    model = sm.OLS(y, X).fit()
    betahat = np.linalg.inv(X.T @ X) @ X.T @ y
    k = len(betahat)
    df = T - k

    # --- 5️⃣ Initialize results dictionary ---
    results = {}

    # --- 6️⃣ Compute OLS quantities ---

    residuals = (y - X @ betahat)
    SSR = np.dot(residuals,residuals)
    SST = np.dot(y - np.mean(y), y -np.mean(y))
    R2_ols = 1 - SSR / SST
    R2_adj_ols = 1 - (SSR / df) / (SST / (T-1))
    s2_ols = SSR / df
    cov_beta_ols = s2_ols * np.linalg.inv(X.T @ X)
    se_beta_ols = np.sqrt(np.diag(cov_beta_ols))
    t_stats_ols = betahat / se_beta_ols
    p_value_ols = 2 * (1 - t.cdf(np.abs(t_stats_ols),df))
    z_95 = norm.ppf(1-0.05/2)
    ci_asymp_ols_95 = np.column_stack([betahat - z_95*se_beta_ols, betahat + z_95*se_beta_ols])


    results['OLS'] = {
        'betahat': betahat,
        'residuals': residuals,
        's2': s2_ols,
        'R2': R2_ols,
        'R2_adj': R2_adj_ols,
        'cov_beta': cov_beta_ols,
        'se_beta': se_beta_ols,
        't_stats': t_stats_ols,
        'p_value': p_value_ols,
        'conf_int_95': ci_asymp_ols_95,
    }

    # --- 7️⃣ Compute Newey–West quantities ---
    for m in bandwidths:
        cov_nw = cov_hac(model, nlags=m)
        se_nw = np.sqrt(np.diag(cov_nw))
        t_nw = betahat / se_nw
        p_nw = 2 * (1 - norm.cdf(np.abs(t_nw)))
        ci_nw_95 = np.column_stack([betahat - z_95*se_nw, betahat + z_95*se_nw])

        results[f'NW({m})'] = {
            'var_sandwich': cov_nw,
            'cov_beta': np.diag(cov_nw),
            'conf_int_95': ci_nw_95,
            'p_value': p_nw
        }

    return results



In [61]:
results = one_replication(T, phi, betas_true, bandwidths)

In [62]:
np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
results["OLS"]["conf_int_95"]

array([[-0.46, -0.0713],
       [0.74, 1.16],
       [0.171, 0.57],
       [-0.77, -0.369]])

In [63]:
for m in bandwidths:
  print(f"NW({m})\n", results[f"NW({m})"]["conf_int_95"])

NW(0)
 [[-0.468 -0.0636]
 [0.76 1.14]
 [0.156 0.585]
 [-0.763 -0.376]]
NW(1)
 [[-0.493 -0.0382]
 [0.777 1.12]
 [0.155 0.586]
 [-0.756 -0.382]]
NW(4)
 [[-0.515 -0.0168]
 [0.791 1.11]
 [0.141 0.6]
 [-0.744 -0.395]]
NW(4)
 [[-0.515 -0.0168]
 [0.791 1.11]
 [0.141 0.6]
 [-0.744 -0.395]]


In [64]:
np.set_printoptions(formatter={'float_kind': '{:0.3e}'.format})
results["OLS"]["p_value"]

array([8.695e-03, 3.464e-14, 4.520e-04, 2.372e-07])

In [65]:
for m in bandwidths:
  print(results[f"NW({m})"]["p_value"])

[9.980e-03 0.000e+00 7.046e-04 7.817e-09]
[2.207e-02 0.000e+00 7.646e-04 2.398e-09]
[3.639e-02 0.000e+00 1.523e-03 1.710e-10]
[3.639e-02 0.000e+00 1.523e-03 1.710e-10]


In [66]:
def montecarlo_analysis(T, phi, betas, bandwidths, n_rep=10000):
    """Run Monte Carlo to check CI coverage and p-values"""
    p = len(betas)
    betahat_distr = np.zeros((n_rep,p))

    for i in trange(n_rep):
        res = one_replication(T, phi, betas, bandwidths)
        betahat_distr[i,:] = res['OLS']['betahat']
    return np.array(betahat_distr)

# Example usage
T = 200
bandwidths = [int(4 * (T/100)**(2/9))]

betahat_distr = montecarlo_analysis(T, phi, betas_true, bandwidths)
np.set_printoptions(formatter={'float_kind': '{:0.3}'.format})
print(betahat_distr)

  0%|          | 0/10000 [00:00<?, ?it/s]

[[0.171 0.98 0.449 -0.395]
 [-0.107 1.1 0.589 -0.463]
 [-0.177 0.897 0.458 -0.59]
 ...
 [0.0318 0.94 0.501 -0.451]
 [-0.161 0.986 0.448 -0.485]
 [-0.0733 1.05 0.491 -0.368]]


In [83]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

k = betahat_distr.shape[1]
n_rows, n_cols = 2, 2

# --- Compute global y-axis limit for consistent density scale ---
all_counts = []
for i in range(k):
    hist, _ = np.histogram(betahat_distr[:, i], bins=100, density=True)
    all_counts.append(np.max(hist))
y_max = max(all_counts) * 1.1

# --- Create subplots ---
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"β{i}" for i in range(k)],
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

# --- Add each β̂ histogram + vertical lines ---
for i in range(k):
    row = i // n_cols + 1
    col = i % n_cols + 1
    beta_true = betas_true[i]
    beta_mean = np.mean(betahat_distr[:, i])

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=betahat_distr[:, i],
            nbinsx=100,
            histnorm='probability density',
            marker_color='rgba(0, 90, 180, 0.6)',
            showlegend=False
        ),
        row=row, col=col
    )

    # Vertical lines: true β (green) and sample mean (red dashed)
    fig.add_vline(
        x=beta_true,
        line=dict(color='green', width=3),
        row=row, col=col
    )
    fig.add_vline(
        x=beta_mean,
        line=dict(color='red', width=2, dash='dash'),
        row=row, col=col
    )

# --- Add dummy traces for legend ---
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color='green', width=3),
                         name='True β'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color='red', width=2, dash='dash'),
                         name='Sample Mean β_OLS'))

# --- Style adjustments ---
fig.update_xaxes(title_text="β_OLS value", title_font=dict(size=14))
fig.update_yaxes(range=[0, y_max], title_text="Density", title_font=dict(size=14))

fig.update_layout(
    height=900,
    width=1150,
    title={
        'text': "Monte Carlo Distributions of β_OLS Estimates",
        'x': 0.5,  # center title
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=22, family="Arial Bold")
    },
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(size=15, family="Arial", color="black"),
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    margin=dict(l=70, r=70, t=100, b=100)
)

fig.show()
